# 🎬 OpenSource Clipping — Paste Config Mode

**One cell does it all.** Paste your `.env`-style config below and hit play.

Supported keys (any line left blank uses the default):
```env
VIDEO_URL=https://www.youtube.com/watch?v=Dc4_aBFAYWE
MAX_CLIPS=5
ASPECT_RATIO=9:16
AI_PROVIDER=ollama
AI_MODEL=llama3.1
OLLAMA_URL=http://localhost:11434
OLLAMA_API_KEY=
GOOGLE_API_KEY=
WHISPER_MODEL=large-v3
VIDEO_PRESET=ultrafast
DISABLE_BGM=True
DISABLE_BROLL=True
```

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ONE CELL — SETUP + PASTE CONFIG + RUN + STREAM ALL OUTPUT
# ═══════════════════════════════════════════════════════════════
import os, subprocess, sys, shutil, re, time, json
from datetime import datetime
from pathlib import Path

# ── Paste .env-style config below, then run this cell ──
ENV_CONFIG = """
# paste config here
VIDEO_URL=
"""

# ── DEFAULT CONFIG (do not delete) ──
DEFAULTS = {
    "VIDEO_URL": "",
    "SOURCE_PLATFORM": "youtube",
    "AI_PROVIDER": "ollama",
    "AI_MODEL": "llama3.1",
    "OLLAMA_URL": "http://localhost:11434",
    "OLLAMA_FALLBACK_URL": "http://localhost:11434",
    "OLLAMA_FALLBACK_MODEL": "llama3.1",
    "GEMINI_MODEL": "gemini-3-flash-preview",
    "OLLAMA_API_KEY": "",
    "GOOGLE_API_KEY": "",
    "NVIDIA_API_KEY": "",
    "MAX_CLIPS": "5",
    "ASPECT_RATIO": "9:16",
    "RENDER_HEIGHT": "1080",
    "SOURCE_HEIGHT": "max",
    "FONT_STYLE": "HORMOZI",
    "FACE_DETECTOR": "mediapipe",
    "YOLO_SIZE": "8m",
    "ENABLE_SUBTITLES": "True",
    "USE_KARAOKE_EFFECT": "True",
    "WORDS_PER_SUBTITLE": "5",
    "USE_YT_TRANSCRIPT_API": "True",
    "USE_DLP_SUBS": "True",
    "WHISPER_MODEL": "large-v3",
    "WHISPER_DEVICE": "auto",
    "WHISPER_COMPUTE_TYPE": "float16",
    "VIDEO_PRESET": "ultrafast",
    "VIDEO_SCALE_ALGO": "bicubic",
    "VIDEO_CQ": "23",
    "VIDEO_CRF": "20",
    "VIDEO_SHARPEN": "False",
    "ENABLE_COLAB_CLEANUP": "True",
    "DISABLE_BGM": "True",
    "DISABLE_BROLL": "True",
    "DISABLE_HOOK_GLITCH": "False",
    "ENABLE_HOOK_V2": "False",
    "HOOK_DURATION": "3",
    "ENABLE_SPLIT_SCREEN": "False",
    "SPLIT_TRIGGER": "diarization",
    "ENABLE_CAMERA_SWITCH": "False",
    "DISABLE_SEGMENT_TRIM": "False",
    "AGGRESSIVE_SILENCE_TRIM": "False",
    "HF_TOKEN": "",
}

def parse_env(text: str) -> dict:
    result = {}
    for raw in text.strip().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        result[key.strip()] = value.strip()
    return result

def to_bool(v):
    return str(v).strip().lower() in ("true", "1", "yes", "on")

def to_int(v, default=0):
    try:
        return int(str(v).strip())
    except Exception:
        return default

user_cfg = parse_env(ENV_CONFIG)
cfg = dict(DEFAULTS)
cfg.update(user_cfg)

VIDEO_URL = cfg["VIDEO_URL"]
SOURCE_PLATFORM = cfg["SOURCE_PLATFORM"]
AI_PROVIDER = cfg["AI_PROVIDER"]
AI_MODEL = cfg["AI_MODEL"]
OLLAMA_URL = cfg["OLLAMA_URL"]
OLLAMA_FALLBACK_URL = cfg["OLLAMA_FALLBACK_URL"]
OLLAMA_FALLBACK_MODEL = cfg["OLLAMA_FALLBACK_MODEL"]
GEMINI_MODEL = cfg["GEMINI_MODEL"]
OLLAMA_API_KEY = cfg["OLLAMA_API_KEY"]
GOOGLE_API_KEY = cfg["GOOGLE_API_KEY"]
NVIDIA_API_KEY = cfg["NVIDIA_API_KEY"]
MAX_CLIPS = to_int(cfg["MAX_CLIPS"], 5)
ASPECT_RATIO = cfg["ASPECT_RATIO"]
RENDER_HEIGHT = cfg["RENDER_HEIGHT"]
SOURCE_HEIGHT = cfg["SOURCE_HEIGHT"]
FONT_STYLE = cfg["FONT_STYLE"]
FACE_DETECTOR = cfg["FACE_DETECTOR"]
YOLO_SIZE = cfg["YOLO_SIZE"]
ENABLE_SUBTITLES = to_bool(cfg["ENABLE_SUBTITLES"])
USE_KARAOKE_EFFECT = to_bool(cfg["USE_KARAOKE_EFFECT"])
WORDS_PER_SUBTITLE = to_int(cfg["WORDS_PER_SUBTITLE"], 5)
USE_YT_TRANSCRIPT_API = to_bool(cfg["USE_YT_TRANSCRIPT_API"])
USE_DLP_SUBS = to_bool(cfg["USE_DLP_SUBS"])
WHISPER_MODEL = cfg["WHISPER_MODEL"]
WHISPER_DEVICE = cfg["WHISPER_DEVICE"]
WHISPER_COMPUTE_TYPE = cfg["WHISPER_COMPUTE_TYPE"]
VIDEO_PRESET = cfg["VIDEO_PRESET"]
VIDEO_SCALE_ALGO = cfg["VIDEO_SCALE_ALGO"]
VIDEO_CQ = to_int(cfg["VIDEO_CQ"], 23)
VIDEO_CRF = to_int(cfg["VIDEO_CRF"], 20)
VIDEO_SHARPEN = to_bool(cfg["VIDEO_SHARPEN"])
ENABLE_COLAB_CLEANUP = to_bool(cfg["ENABLE_COLAB_CLEANUP"])
DISABLE_BGM = to_bool(cfg["DISABLE_BGM"])
DISABLE_BROLL = to_bool(cfg["DISABLE_BROLL"])
DISABLE_HOOK_GLITCH = to_bool(cfg["DISABLE_HOOK_GLITCH"])
ENABLE_HOOK_V2 = to_bool(cfg["ENABLE_HOOK_V2"])
HOOK_DURATION = to_int(cfg["HOOK_DURATION"], 3)
ENABLE_SPLIT_SCREEN = to_bool(cfg["ENABLE_SPLIT_SCREEN"])
SPLIT_TRIGGER = cfg["SPLIT_TRIGGER"]
ENABLE_CAMERA_SWITCH = to_bool(cfg["ENABLE_CAMERA_SWITCH"])
DISABLE_SEGMENT_TRIM = to_bool(cfg["DISABLE_SEGMENT_TRIM"])
AGGRESSIVE_SILENCE_TRIM = to_bool(cfg["AGGRESSIVE_SILENCE_TRIM"])
HF_TOKEN = cfg["HF_TOKEN"]

# ── Basic validation ──
errors = []
if not VIDEO_URL.strip():
    errors.append("VIDEO_URL wajib diisi.")
elif not VIDEO_URL.strip().startswith(("http://", "https://")):
    errors.append("VIDEO_URL harus diawali http:// atau https://.")
if AI_PROVIDER == "gemini" and not GOOGLE_API_KEY.strip():
    errors.append("GOOGLE_API_KEY wajib jika AI_PROVIDER='gemini'.")
if AI_PROVIDER == "nvidia" and not NVIDIA_API_KEY.strip():
    errors.append("NVIDIA_API_KEY wajib jika AI_PROVIDER='nvidia'.")
if errors:
    print("❌ Validation failed:")
    for e in errors:
        print(f"  • {e}")
    raise SystemExit("Validation failed.")

print("✅ Config loaded. Overridden keys:")
for k in sorted(user_cfg.keys()):
    display = user_cfg[k] if "KEY" not in k.upper() else "***"
    print(f"  • {k} = {display}")
if not user_cfg:
    print("  (using all defaults)")

# ── Repo paths ──
REPO_URL = "https://github.com/troublescope/Ashortai.git"
CLONE_DIR = "/content/opensource-clipping"
Path(CLONE_DIR).mkdir(parents=True, exist_ok=True)

# ── Mount Drive (optional) ──
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

# ── Clone / update repo ──
if os.path.isdir(f"{CLONE_DIR}/.git"):
    print("\n📁 Repo exists — pulling latest…")
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only"], check=False)
else:
    print("\n⬇️  Cloning repo…")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)

os.chdir(CLONE_DIR)

# ── Install dependencies ──
print("⏳ Installing dependencies…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("✅ Dependencies installed.")

# ── Restore models from Drive cache if available ──
try:
    DRIVE_CACHE = Path("/content/drive/MyDrive/osc_cache")
    DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
    for m in ["blaze_face_full_range.tflite", "face_yolov8m.pt", "face_yolov8n.pt", "face_yolov8s.pt"]:
        src, dst = DRIVE_CACHE / m, Path(CLONE_DIR) / m
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            print(f"📥 Restored {m}")
except Exception:
    pass

# ── Write .env file ──
env_lines = []
if OLLAMA_API_KEY.strip(): env_lines.append(f"OLLAMA_API_KEY={OLLAMA_API_KEY.strip()}")
if GOOGLE_API_KEY.strip(): env_lines.append(f"GOOGLE_API_KEY={GOOGLE_API_KEY.strip()}")
if NVIDIA_API_KEY.strip(): env_lines.append(f"NVIDIA_API_KEY={NVIDIA_API_KEY.strip()}")
if HF_TOKEN.strip(): env_lines.append(f"HF_TOKEN={HF_TOKEN.strip()}")
Path(CLONE_DIR, ".env").write_text("\n".join(env_lines) + "\n", encoding="utf-8")
print("🔐 .env written.")

# ── Unique output folder ──
def _slugify(s: str) -> str:
    s = s.split("?")[0].split("/")[-1]
    s = re.sub(r"[^\w\-_. ]", "", s).strip()
    return s[:40] or "clip"

url = VIDEO_URL.strip()
output_slug = f"{_slugify(url)}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = Path(CLONE_DIR) / "outputs" / output_slug
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Output target: {OUTPUT_DIR}")

# ── Build command ──
cmd = [
    sys.executable, "main.py",
    "--url", url,
    "--source", SOURCE_PLATFORM,
    "--clips", str(MAX_CLIPS),
    "--ratio", ASPECT_RATIO,
    "--render-height", str(RENDER_HEIGHT),
    "--source-height", str(SOURCE_HEIGHT),
    "--font-style", FONT_STYLE,
    "--face-detector", FACE_DETECTOR,
    "--yolo-size", YOLO_SIZE,
    "--ai-provider", AI_PROVIDER,
    "--gemini-model", GEMINI_MODEL,
    "--ollama-url", OLLAMA_URL,
    "--ollama-model", AI_MODEL,
    "--ollama-fallback-url", OLLAMA_FALLBACK_URL,
    "--ollama-fallback-model", OLLAMA_FALLBACK_MODEL,
    "--whisper-model", WHISPER_MODEL,
    "--whisper-device", WHISPER_DEVICE,
    "--whisper-compute-type", WHISPER_COMPUTE_TYPE,
    "--video-preset", VIDEO_PRESET,
    "--video-scale-algo", VIDEO_SCALE_ALGO,
    "--video-cq", str(VIDEO_CQ),
    "--video-crf", str(VIDEO_CRF),
    "--words-per-sub", str(WORDS_PER_SUBTITLE),
    "--hook-duration", str(HOOK_DURATION),
]

def add_flag(flag, cond):
    if cond:
        cmd.append(flag)

add_flag("--colab-cleanup", ENABLE_COLAB_CLEANUP)
add_flag("--use-yt-transcript-api", USE_YT_TRANSCRIPT_API)
add_flag("--use-dlp-subs", USE_DLP_SUBS)
add_flag("--no-bgm", DISABLE_BGM)
add_flag("--no-broll", DISABLE_BROLL)
add_flag("--no-subs", not ENABLE_SUBTITLES)
add_flag("--no-karaoke", not USE_KARAOKE_EFFECT)
add_flag("--no-hook", DISABLE_HOOK_GLITCH)
add_flag("--camera-switch", ENABLE_CAMERA_SWITCH)
add_flag("--hook-v2", ENABLE_HOOK_V2)
add_flag("--no-segment-trim", DISABLE_SEGMENT_TRIM)
add_flag("--silence-trim", AGGRESSIVE_SILENCE_TRIM)
add_flag("--video-sharpen", VIDEO_SHARPEN)
add_flag("--split-screen", ENABLE_SPLIT_SCREEN)

if ENABLE_SPLIT_SCREEN:
    cmd += ["--split-trigger", SPLIT_TRIGGER]

# ── Run pipeline and print all output live ──
print("\n🚀 Starting pipeline…")
print("URL:", url)
print("Command:", " ".join(cmd))
print("─" * 60)

start = time.time()
process = subprocess.Popen(
    cmd,
    cwd=CLONE_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
process.wait()
print("─" * 60)

if process.returncode != 0:
    print(f"❌ Pipeline exited with code {process.returncode}")
else:
    print("✅ Pipeline finished successfully.")

elapsed = time.time() - start
print(f"⏱️ Total time: {elapsed/60:.1f} minutes")

# ── Collect outputs into unique folder ──
base_out = Path(CLONE_DIR) / "outputs"
if base_out.exists():
    moved = 0
    for f in sorted(base_out.iterdir()):
        if f.is_file():
            try:
                shutil.move(str(f), str(OUTPUT_DIR / f.name))
                moved += 1
            except Exception as e:
                print(f"⚠️ Could not move {f.name}: {e}")
    print(f"\n📁 {moved} file(s) moved to: {OUTPUT_DIR}")

    files = sorted(OUTPUT_DIR.glob("*_ready.mp4")) + sorted(OUTPUT_DIR.glob("*.jpg"))
    if files:
        print(f"\n📦 Output files ({len(files)}):")
        for f in files:
            print(f"   • {f.name} ({f.stat().st_size/1024/1024:.1f} MB)")
    if list(OUTPUT_DIR.iterdir()):
        zip_path = OUTPUT_DIR / "outputs.zip"
        shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=str(OUTPUT_DIR))
        print(f"\n🗜️ outputs.zip → {zip_path.stat().st_size/1024/1024:.1f} MB")
        try:
            from google.colab import files
            files.download(str(zip_path))
        except Exception:
            pass
else:
    print("⚠️ outputs/ directory not found.")
